## Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd
import os
from io import BytesIO
from datetime import datetime
import sys
sys.path.append("../aws_utils")
import aws_utils

In [0]:
env = aws_utils.get_env_prefix()

In [0]:
spark.sql(f"""
          CREATE VOLUME IF NOT EXISTS `{env}-cdh-delivery`.tiresias.exports
          """)

In [0]:
run_date = datetime.now().strftime("%Y-%m-%d")

# Convert to datetime object
date_obj = datetime.strptime(run_date, "%Y-%m-%d")

# Reformat to desired format (YYYYMMDD)
formatted_date = date_obj.strftime("%Y%m%d")

print(formatted_date)

In [0]:
output_path = f"/Volumes/{env}-cdh-delivery/tiresias/exports/{formatted_date}/"

In [0]:
def create_single_text_file(_df, _filename):
    # Your DataFrame and output folder
    _df = _df.select("RawData")  # or any df with one string column

    # Skip if only header row (no detail rows)
    if _df.count() <= 1:
        print(f"Skipping {_filename} — no detail rows found.")
        return
    
    # This is the FINAL path where the file will end up (e.g., .../exports/20251110/Individual_01.txt)
    final_file_path = os.path.join(output_path, _filename)
    # --- CHANGE 1: Define a unique temporary path for Spark to write to ---
    # We'll write to a temporary sub-directory *inside* your main folder
    temp_write_path = os.path.join(output_path, f"__temp_{_filename}")

    # Step 1: Write with one partition to the *temporary* path
    # This change is CRITICAL. It writes to a new, unique folder
    # so mode("overwrite") doesn't delete your other files.
    _df.coalesce(1).write.mode("overwrite").text(temp_write_path) # <-- CHANGED

    # Step 2: Use dbutils.fs to list files in the *temporary* folder
    files = dbutils.fs.ls(temp_write_path) # <-- CHANGED

    # Step 3: Find the part file Spark wrote (starts with "part-" and ends with ".txt")
    part_file_path = None
    for file in files:
        if file.name.startswith("part-") and file.name.endswith(".txt"):
            part_file_path = file.path
            break

    if not part_file_path:
        raise Exception(f"No part file found in {temp_write_path}!") # <-- (Error message updated)

    # Step 4: Move/rename part file to desired *final* file path
    # This moves the file *out* of the temp folder and into the main output_path
    dbutils.fs.mv(part_file_path, final_file_path)
    # --- CHANGE 2: Clean up the *temporary directory* ---
    # Step 5: Delete the entire temporary directory and its contents (like _SUCCESS)
    dbutils.fs.rm(temp_write_path, recurse=True) # <-- REPLACED original Step 5

    print(f"Single text file created at {final_file_path}")

##QUERY FOR CONSUMER

In [0]:
def dataframe_creation_individuals(_export_partition, _env, _portfolio_filter='EXCLUDE_GALENE'):
    # _portfolio_filter: 'EXCLUDE_GALENE' = all except GALENE, 'ONLY_GALENE' = only GALENE
    if _portfolio_filter == 'ONLY_GALENE':
        portfolio_clause = "AND portfolio = 'GALENE'"
    else:
        portfolio_clause = "AND portfolio != 'GALENE'"

    df_for_export = spark.sql(f"""
SELECT
  LPAD('', 6, 0) ||
  'CPA' ||
  LPAD(YEAR(current_date()), 4, 0) ||
  LPAD(MONTH(current_date()), 2, 0) ||
  LPAD(DAY(current_date()), 2, 0) ||
  LPAD(COUNT(*), 8, 0) ||
  LPAD(REPLACE(COALESCE(SUM(`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΔΑΝΕΙΟΥ`), 0), '.', ''), 15, 0) ||
  RPAD('', 409, ' ') AS RawData
from `{_env}-cdh-delivery`.tiresias.export_consumer
where ExportPartition = '{_export_partition}' {portfolio_clause}

UNION ALL

SELECT
    LPAD(`ΑΥΞΩΝ_ΑΡΙΘΜΟΣ`, 6, 0) ||
    '0001' ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`, ''), 16, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`), 0), 2, 0) ||
    LPAD(COALESCE(`ΑΡ_ΔΟΣΕΩΝ_ΔΑΝΕΙΟΥ`, 0), 3, 0) ||
    LPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΚΑΘΥΣΤΕΡΗΣΗΣ`, 0), 2, 0) ||
    LPAD(REPLACE(COALESCE(`ΠΟΣΟ_ΔΟΣΗΣ`, 0), '.', ''), 8, 0) ||
    LPAD(REPLACE(COALESCE(`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΔΑΝΕΙΟΥ`, 0), '.', ''), 12, 0) ||
    LPAD(COALESCE(`ΣΥΧΝΟΤΗΤΑ_ΔΟΣΕΩΝ`, 0), 2, 0) ||
    LPAD(REPLACE(COALESCE(`ΕΝΗΜΕΡΟ_ΥΠΟΛΟΙΠΟ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(COALESCE(`ΥΠΟΛΟΙΠΟ_ΣΕ_ΚΑΘΥΣΤΕΡΗΣΗ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(COALESCE(`ΑΡΝΗΤΙΚΟ_ΥΠΟΛΟΙΠΟ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(COALESCE(`ΣΥΝΟΛΟ_ΟΦΕΙΛΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΕΓΚΡΙΣΗΣ_ΔΑΝΕΙΟΥ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΕΓΚΡΙΣΗΣ_ΔΑΝΕΙΟΥ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΕΓΚΡΙΣΗΣ_ΔΑΝΕΙΟΥ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`), 0), 2, 0) ||
    CASE WHEN `{_env}-cdh`.derivative.is_valid_afm(`ΑΦΜ`) = FALSE THEN '000000000' ELSE `ΑΦΜ` END ||
    RPAD(COALESCE(`ΑΔΤ`, ''), 9, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`), 0), 2, 0) ||
    RPAD('', 9, ' ') ||
    RPAD(COALESCE(`ΕΠΩΝΥΜΟ`, ''), 24, ' ') ||
    RPAD(COALESCE(`ΟΝΟΜΑ`, ''), 6, ' ') ||
    RPAD(COALESCE(`ΠΑΤΡΩΝΥΜΟ`, ''), 3, ' ') ||
    RPAD('', 6, ' ') ||
    RPAD(COALESCE(`ΟΔΟΣ`, ''), 20, ' ') ||
    RPAD(COALESCE(`ΑΡΙΘΜΟΣ`, ''), 4, ' ') ||
    RPAD(COALESCE(`ΠΟΛΗ`, ''), 20, ' ') ||
    CASE WHEN REPLACE(`ΤΑΧ_ΚΩΔΙΚΑΣ`, ' ', '') RLIKE '^[0-9]+$' THEN RPAD(REPLACE(`ΤΑΧ_ΚΩΔΙΚΑΣ`, ' ', ''), 5, '0') ELSE '00000' END ||
    LPAD(REPLACE(COALESCE(`ΣΥΝΟΛΙΚΟ_ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΣΤΕΓ_ΔΑΝΕΙΟΥ`, 0), '.', ''), 12, 0) ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΔΑΝΕΙΟΥ`, ''), 16, ' ') ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΕΠΙΒΕΒΑΙΩΣΗΣ`, ''), 2, ' ') ||
    RPAD(COALESCE(`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`, ''), 2, ' ') ||
    RPAD(COALESCE(`ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`, ''), 2, ' ') ||
    RPAD(COALESCE(`ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΡΥΘΜΙΣΗΣ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΜΕΓΙΣΤΗ_ΚΑΘΥΣΤΕΡΗΣΗ_ΡΥΘΜ_ΧΟΡΗΓ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΔΙΑΡΚΕΙΑ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`, ''), 3, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΠΕΡΙΟΔΟΣ_ΧΑΡΙΤΟΣ_ΤΟΚΩΝ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΡΥΘΜΙΣΗΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΡΥΘΜΙΣΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΡΥΘΜΙΣΗΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΝΕΑΣ_ΡΥΘΜΙΣΗΣ_ΒΑΣΕΙ_ΠΕΕ`, ''), 16, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ/ΝΙΑ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ/ΝΙΑ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ/ΝΙΑ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΚΑΝΑΛΙ_ΔΙΑΘΕΣΗΣ_ΧΟΡΗΓΗΣΗΣ`, ''), 3, ' ') ||
    RPAD(COALESCE(`Bank_origin_code`, ''), 3, ' ') ||
    RPAD(COALESCE(`fund`, ''), 20, ' ') ||
    RPAD(COALESCE(`portfolio`, ''), 20, ' ') ||
    RPAD(COALESCE(`ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ`, ''), 50, ' ') ||
    RPAD(COALESCE(`ΑΡΙΘΜΟΣ_ΑΠΟΦΑΣΗΣ`, ''), 50, ' ') ||
    RPAD(COALESCE(`RRE_ΓΙΑ_ΑΠΦ`, ''), 1, ' ') ||
    LPAD(REPLACE(COALESCE(`ΠΟΣΟ_ΟΦΕΙΛΗΣ_ΠΡΟ_ΡΥΘΜΙΣΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(0, 12, 0) ||
    RPAD('', 1, ' ') ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΣΥΝΔΕΔΕΜΕΝΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`, ''), 16, ' ') ||
    RPAD('', 149, ' ') AS RawData
FROM `{_env}-cdh-delivery`.tiresias.export_consumer
where ExportPartition = '{_export_partition}' {portfolio_clause}
""")
    
    return df_for_export

##QUERY FOR BUSINESS

In [0]:
def dataframe_creation_business(_env, _portfolio_filter='EXCLUDE_GALENE'):
    # _portfolio_filter: 'EXCLUDE_GALENE' = all except GALENE, 'ONLY_GALENE' = only GALENE
    if _portfolio_filter == 'ONLY_GALENE':
        portfolio_clause = "WHERE portfolio = 'GALENE'"
    else:
        portfolio_clause = "WHERE portfolio != 'GALENE'"

    df_for_export = spark.sql(f"""
SELECT
  LPAD('', 6, 0) ||
  'CPA' ||
  LPAD(YEAR(current_date()), 4, 0) ||
  LPAD(MONTH(current_date()), 2, 0) ||
  LPAD(DAY(current_date()), 2, 0) ||
  LPAD(COUNT(*), 8, 0) ||
  LPAD(REPLACE(COALESCE(SUM(`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΟΡΙΟ_ΧΟΡΗΓΗΣΗΣ`), 0), '.', ''), 15, 0) ||
  RPAD('', 659, ' ') AS RawData
FROM `{_env}-cdh-delivery`.tiresias.export_business
{portfolio_clause}

UNION ALL 
SELECT 
    LPAD(COALESCE(`ΑΥΞΩΝ_ΑΡΙΘΜΟΣ`, 0), 6, 0) ||
    '0001' ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`, ''), 16, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΟΥ_ΟΡΙΟΥ_ΠΟΣΟΥ_ΕΓΚΡΙΣΗΣ_ΜΕ_ΑΛΛΗ_ΕΠΙΧΕΙΡΗΣΗ`, 0), 2, 0) ||
    LPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΕΓΚΡΙΣΗΣ_ΧΟΡΗΓΗΣΕΩΝ_ΤΗΣ_ΙΔΙΑΣ_ΕΠΙΧΕΙΡΗΣΗΣ`, ''), 16, ' ') ||
    LPAD(COALESCE(`ΑΡ_ΔΟΣΕΩΝ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`, 0), 3, 0) ||
    LPAD(COALESCE(`ΥΠΟΛΕΙΠΟΜΕΝΕΣ_ΔΟΣΕΙΣ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`, 0), 3, 0) ||
    LPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΚΑΘΥΣΤΕΡΗΣΗΣ`, 0), 2, 0) ||
    LPAD(REPLACE(COALESCE(`ΠΟΣΟ_ΔΟΣΗΣ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(COALESCE(`ΠΟΣΟ_ΕΠΟΜΕΝΗΣ_ΕΛΑΧΙΣΤΗΣ_ΚΑΤΑΒΟΛΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(IFNULL(`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΟΡΙΟ_ΧΟΡΗΓΗΣΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(IFNULL(`ΣΥΧΝΟΤΗΤΑ_ΔΟΣΕΩΝ_Η_ΕΛΑΧΙΣΤΩΝ_ΚΑΤΑΒΟΛΩΝ`, 0), 2, 0) ||
    LPAD(REPLACE(IFNULL(`ΠΟΣΟ_ΠΛΗΡΩΜΩΝ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΜΗΝΑ`, 0), '.', ''), 12, 0) ||
    LPAD(COALESCE(YEAR(`ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΚΙΝΗΣΗΣ_ΛΟΓΑΡΙΑΣΜΟΥ`) ,0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΚΙΝΗΣΗΣ_ΛΟΓΑΡΙΑΣΜΟΥ`),0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΚΙΝΗΣΗΣ_ΛΟΓΑΡΙΑΣΜΟΥ`) ,0), 2, 0) ||
    LPAD(REPLACE(IFNULL(`ΕΝΗΜΕΡΟ_ΥΠΟΛΟΙΠΟ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(IFNULL(`ΥΠΟΛΟΙΠΟ_ΣΕ_ΚΑΘΥΣΤΕΡΗΣΗ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(IFNULL(`ΑΡΝΗΤΙΚΟ_ΥΠΟΛΟΙΠΟ`, 0), '.', ''), 12, 0) ||
    LPAD(REPLACE(IFNULL(`ΣΥΝΟΛΟ_ΟΦΕΙΛΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΕΓΚΡΙΣΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`) ,0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΕΓΚΡΙΣΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`),0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΕΓΚΡΙΣΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`)  ,0), 2, 0) ||
    RPAD(IFNULL(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`, ''), 1, ' ') ||
    RPAD(IFNULL(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`, ''), 1, ' ') ||
    RPAD(IFNULL(`ΔΕΙΚΤΗΣ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`) ,0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`),0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`)  ,0), 2, 0) ||
    CASE WHEN `{_env}-cdh`.derivative.is_valid_afm(`ΑΦΜ`) = FALSE THEN '000000000' ELSE `ΑΦΜ` END ||
    RPAD(IFNULL(`ΑΔΤ`, ''), 9, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`) ,0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`),0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`)  ,0), 2, 0) ||
    RPAD('', 8, ' ') ||
    LPAD(IFNULL(`ΦΥΛΟ`, 0), 1, 0) ||
    RPAD(IFNULL(`ΕΠΩΝΥΜΟ`, ''), 24, ' ') ||
    RPAD(IFNULL(`ΟΝΟΜΑ`, ''), 6, ' ') ||
    RPAD(IFNULL(`ΠΑΤΡΩΝΥΜΟ`, ''), 3, ' ') ||
    RPAD(IFNULL(`ΕΠΩΝΥΜΙΑ_ΕΤΑΙΡΙΑΣ`, ''), 58, ' ') ||
    RPAD(IFNULL(`ΔΙΑΚΡΙΤΙΚΟΣ_ΤΙΤΛΟΣ`, ''), 20, ' ') ||
    LPAD(REPLACE(IFNULL(`ΚΥΚΛΟΣ_ΕΡΓΑΣΙΩΝ_ΕΤΑΙΡΙΑΣ`, 0), '.', ''), 12, 0) ||
    LPAD(IFNULL(`ΕΤΟΣ_ΚΥΚΛΟΥ_ΕΡΓΑΣΙΩΝ_ΕΤΑΙΡΙΑΣ`, 0), 4, 0) ||
    RPAD('', 6, ' ') ||
    RPAD(IFNULL(`ΟΔΟΣ`, ''), 20, ' ') ||
    RPAD(IFNULL(`ΑΡΙΘΜΟΣ`, ''), 4, ' ') ||
    RPAD(IFNULL(`ΠΟΛΗ`, ''), 20, ' ') ||
    CASE WHEN REPLACE(`ΤΑΧ_ΚΩΔΙΚΑΣ`, ' ', '') RLIKE '^[0-9]+$' THEN RPAD(REPLACE(`ΤΑΧ_ΚΩΔΙΚΑΣ`, ' ', ''), 5, '0') ELSE '00000' END ||
    LPAD(REPLACE(IFNULL(`ΣΥΝΟΛΙΚΟ_ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(COALESCE(YEAR(`ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`) ,0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`),0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`)  ,0), 2, 0) ||
    RPAD(IFNULL(`ΚΩΔΙΚΟΣ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`, ''), 16, ' ') ||
    LPAD(IFNULL(`ΔΕΙΚΤΗΣ_ΜΕΤΑΒΑΣΗΣ_ΣΤΙΣ_ΜΕΣΑΙΕΣ_Η_ΜΕΓΑΛΕΣ_ΕΠΙΧΕΙΡΗΣΕΙΣ`, 0), 1, 0) ||
    RPAD(IFNULL(`ΚΩΔΙΚΟΣ_ΕΠΙΒΕΒΑΙΩΣΗΣ`, ''), 2, ' ') ||
    RPAD(IFNULL(`ΚΛΑΔΟΣ_ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`, ''), 8, ' ') ||
    LPAD(COALESCE(`ΤΑΞΙΝΟΜΗΣΗ_ΚΛΑΔΟΥ_ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`,0), 2, 0) ||
    LPAD(COALESCE(YEAR(`ΗΜΕΡΟΜΗΝΙΑ_ΕΝΗΜΕΡΩΣΗΣ_ΚΛΑΔΟΥ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜΕΡΟΜΗΝΙΑ_ΕΝΗΜΕΡΩΣΗΣ_ΚΛΑΔΟΥ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜΕΡΟΜΗΝΙΑ_ΕΝΗΜΕΡΩΣΗΣ_ΚΛΑΔΟΥ`), 0), 2, 0) ||
    LPAD(COALESCE(`ΝΟΜΙΚΗ_ΜΟΡΦΗ_ΕΤΑΙΡΕΙΑΣ`,0), 2, 0) ||
    LPAD(COALESCE(`ΕΙΔΟΣ_HAIRCUT_ΛΟΓΩ_ΡΥΘΜΙΣΗΣ`,0), 1, 0) ||
    LPAD(COALESCE(`ΠΟΣΟΣΤΟ_HAIRCUT_ΛΟΓΩ_ΡΥΘΜΙΣΗΣ`,0), 4, 0) ||
    LPAD(COALESCE(`ΔΙΑΡΚΕΙΑ_ΡΥΘΜΙΣΗΣ`,0), 3, 0) ||
    RPAD(COALESCE(`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`,''), 2, ' ') ||
    RPAD(COALESCE(`ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`,''), 1, ' ') ||
    RPAD(COALESCE(`ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`,''), 1, ' ') ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,''), 1, ' ') ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,''), 2, ' ') ||
    RPAD(COALESCE(`ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`,''), 1, ' ') ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΡΥΘΜΙΣΗΣ`,''), 1, ' ') ||
    RPAD(COALESCE(`ΜΕΓΙΣΤΗ_ΚΑΘΥΣΤΕΡΗΣΗ_ΡΥΘΜ_ΧΟΡΗΓ`,''), 1, ' ') ||
    RPAD(COALESCE(`ΔΙΑΡΚΕΙΑ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,''), 3, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΠΕΡΙΟΔΟΣ_ΧΑΡΙΤΟΣ_ΤΟΚΩΝ`, ''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΡΥΘΜΙΣΗΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΡΥΘΜΙΣΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΡΥΘΜΙΣΗΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,''), 1, ' ') ||
    RPAD(COALESCE(`ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,''), 1, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`), 0), 2, 0) ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΝΕΑΣ_ΡΥΘΜΙΣΗΣ_βάσει_ΠΕΕ`,''), 16, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`), 0), 2, 0) ||
    RPAD(COALESCE(`Bank_origin_code`, ''), 3, ' ') ||
    RPAD(COALESCE(`fund`, ''), 20, ' ') ||
    RPAD(COALESCE(`portfolio`, ''), 20, ' ') ||
    LPAD(COALESCE(YEAR(`ΗΜ_ΜΕΤΑΒΟΛΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`), 0), 4, 0) ||
    LPAD(COALESCE(MONTH(`ΗΜ_ΜΕΤΑΒΟΛΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`), 0), 2, 0) ||
    LPAD(COALESCE(DAY(`ΗΜ_ΜΕΤΑΒΟΛΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`),0), 2, 0) ||
    RPAD(COALESCE(`Αριθμός_Εργαζομένων`, ''), 6, ' ') ||
    RPAD(COALESCE(`ΜΕΓΕΘΟΣ_ΕΠΙΧΕΙΡΗΣΗΣ`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΌΡΙΟ_ΕΓΚΡΙΣΗΣ_ΑΠΟ_ΙΔΙΑ_ΚΕΦΑΛΑΙΑ_ΤΗΣ_ΤΡΑΠΕΖΑΣ`, ''), 12, ' ') ||
    RPAD(COALESCE(`ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΑΠΟ_ΙΔΙΑ_ΚΕΦΑΛΑΙΑ_ΤΗΣ_ΤΡΑΠΕΖΑΣ`, ''), 12, ' ') ||
    RPAD(COALESCE(`ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`, ''), 1, ' ') ||
    RPAD(COALESCE(`Αριθμός_Σύμβασης_κοινοπρακτικού_δανείου`, ''), 100, ' ') ||
    RPAD(COALESCE(`Συνολικό_εγκριθέν_ποσό_της_σύμβασης_κοινοπρακτικού_δανείου`, ''), 20, ' ') ||
    RPAD(COALESCE(`Δείκτης_ποσοστού_συμμετοχής_κοινοπρακτικού_δανείου`, ''), 1, ' ') ||
    RPAD(COALESCE(`ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ`, ''), 50, ' ') ||
    RPAD(COALESCE(`ΑΡΙΘΜΟΣ_ΑΠΟΦΑΣΗΣ`, ''), 50, ' ') ||
    RPAD(COALESCE(`RRE_ΓΙΑ_ΑΠΦ`, ''), 1, ' ') ||
    LPAD(REPLACE(COALESCE(`ΠΟΣΟ_ΟΦΕΙΛΗΣ_ΠΡΟ_ΡΥΘΜΙΣΗΣ`, 0), '.', ''), 12, 0) ||
    LPAD(0, 12, 0) ||
    RPAD('', 1, ' ') ||
    RPAD(COALESCE(`ΚΩΔΙΚΟΣ_ΣΥΝΔΕΔΕΜΕΝΟΥ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`, ''), 16, ' ') ||
    RPAD('', 3, ' ') AS RawData
FROM `{_env}-cdh-delivery`.tiresias.export_business
{portfolio_clause}

""")
    
    return df_for_export

##EXPORT INDIVIDUALS 

In [0]:
df_individual_1 = dataframe_creation_individuals(1, env)
df_individual_2 = dataframe_creation_individuals(2, env)

filename_part_1 = "Individual_01.txt"
filename_part_2 = "Individual_02.txt"

# GALENE separate exports
df_individual_1_gne = dataframe_creation_individuals(1, env, 'ONLY_GALENE')
df_individual_2_gne = dataframe_creation_individuals(2, env, 'ONLY_GALENE')

filename_part_1_gne = "Individual_01_GNE.txt"
filename_part_2_gne = "Individual_02_GNE.txt"

In [0]:
create_single_text_file(df_individual_1, filename_part_1)

In [0]:
create_single_text_file(df_individual_2, filename_part_2)

In [0]:
# GALENE Individual exports
create_single_text_file(df_individual_1_gne, filename_part_1_gne)
create_single_text_file(df_individual_2_gne, filename_part_2_gne)

##EXPORT BUSINESS 

In [0]:
df_business = dataframe_creation_business(env)

filename_business = "Business.txt"

# GALENE separate export
df_business_gne = dataframe_creation_business(env, 'ONLY_GALENE')

filename_business_gne = "Business_GNE.txt"

In [0]:
create_single_text_file(df_business, filename_business)

In [0]:
# GALENE Business export
create_single_text_file(df_business_gne, filename_business_gne)

##FINAL ZIP - not used

In [0]:
# Convert to datetime object
date_obj = datetime.strptime(formatted_date, "%Y%m%d")

# Format to desired output
zip_formatted_date = date_obj.strftime("%Y_%m_%d")

print(zip_formatted_date)  # Output: 2025_05_26

In [0]:
zip_name = f'Tiresias_{zip_formatted_date}.zip'
print(zip_name)

In [0]:
'''
import zipfile
import os

# --- 1. Define Paths ---
# Use the Volume path for dbutils
source_dir_volume = output_path.rstrip('/')
# Use the /dbfs/ path for zipfile
final_dbfs_zip_path = f"/dbfs{output_path.rstrip('/')}/{zip_name}"
print(f"Preparing to create zip file at: {final_dbfs_zip_path}")

# --- 2. Read file list using dbutils ---
files_to_zip = []
print("Scanning for files using dbutils...")

file_list = dbutils.fs.ls(source_dir_volume)
for file_info in file_list:
    file_name = file_info.name
    
    # Skip directories or the zip file itself
    if file_info.isDir() or file_name == zip_name:
        continue
        
    # Get the /dbfs/ path, which zipfile needs for writing
    full_dbfs_path = f"/dbfs{file_info.path.replace('dbfs:', '')}"
    relative_path = file_name
    
    files_to_zip.append((full_dbfs_path, relative_path))
    print(f"  + Found: {relative_path} (at {full_dbfs_path})")

# --- 3. Write the zip file ---
if not files_to_zip:
    print("⚠️ No files found to zip.")
else:
    print(f"\nCreating zip file with {len(files_to_zip)} files...")
    
    # This is the step that will fail on Serverless
    with zipfile.ZipFile(final_dbfs_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for full_path, relative_path in files_to_zip:
            print(f"  Adding: {relative_path}")
            zipf.write(full_path, arcname=relative_path)

    print(f"\n✅ ZIP file successfully created at: {final_dbfs_zip_path}")
    '''